# 08 - Model Comparison Report

Read MLflow runs, compare the best classification and BP regression models, and write a Markdown summary report.

In [ ]:
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

try:
    dbutils.widgets.text('classification_experiment', '/Shared/CardioTwin_AI_Classification')
    dbutils.widgets.text('regression_experiment', '/Shared/CardioTwin_AI_BP_Regression')
    dbutils.widgets.text('output_report_path', '../reports/model_metrics_summary.md')
    classification_experiment = dbutils.widgets.get('classification_experiment')
    regression_experiment = dbutils.widgets.get('regression_experiment')
    output_report_path = dbutils.widgets.get('output_report_path')
except Exception:
    classification_experiment = 'CardioTwin_AI_Classification'
    regression_experiment = 'CardioTwin_AI_BP_Regression'
    output_report_path = '../reports/model_metrics_summary.md'

client = MlflowClient()

def load_runs(experiment_name):
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        return pd.DataFrame()
    return mlflow.search_runs(experiment_ids=[experiment.experiment_id])

classification_runs = load_runs(classification_experiment)
regression_runs = load_runs(regression_experiment)

if not classification_runs.empty and 'metrics.f1_weighted' in classification_runs.columns:
    best_classification = classification_runs.sort_values('metrics.f1_weighted', ascending=False).iloc[0]
else:
    best_classification = None

if not regression_runs.empty and 'metrics.rmse' in regression_runs.columns:
    best_regression = regression_runs.sort_values('metrics.rmse', ascending=True).iloc[0]
else:
    best_regression = None

lines = [
    '# CardioTwin AI Model Metrics Summary',
    '',
    'This report is generated from MLflow experiment runs in Databricks.',
    '',
    '## Best Classification Model',
]

if best_classification is None:
    lines.append('No classification runs were found. Run notebook 05 first.')
else:
    lines.extend([
        f"- Model: {best_classification.get('tags.mlflow.runName', 'unknown')}",
        f"- Accuracy: {best_classification.get('metrics.accuracy', float('nan')):.4f}",
        f"- Precision weighted: {best_classification.get('metrics.precision_weighted', float('nan')):.4f}",
        f"- Recall weighted: {best_classification.get('metrics.recall_weighted', float('nan')):.4f}",
        f"- F1 weighted: {best_classification.get('metrics.f1_weighted', float('nan')):.4f}",
        '- Selection reason: highest weighted F1 score, which balances performance across the three cardiovascular status classes.',
    ])

lines.extend(['', '## Best Blood Pressure Regression Model'])

if best_regression is None:
    lines.append('No regression runs were found. Run notebook 06 first.')
else:
    lines.extend([
        f"- Model: {best_regression.get('tags.mlflow.runName', 'unknown')}",
        f"- MAE: {best_regression.get('metrics.mae', float('nan')):.4f}",
        f"- RMSE: {best_regression.get('metrics.rmse', float('nan')):.4f}",
        f"- R2 score: {best_regression.get('metrics.r2_score', float('nan')):.4f}",
        '- Selection reason: lowest RMSE, which penalizes larger blood-pressure prediction errors more strongly.',
    ])

lines.extend([
    '',
    '## Notes',
    '- The bundled CSV is synthetic and small. Reported metrics are useful for checking the pipeline, not for clinical claims.',
    '- For a real study, use more patients, patient-level train/test splitting, and external validation.',
])

report_text = '\n'.join(lines) + '\n'
report_path = Path(output_report_path)
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report_text, encoding='utf-8')

print(report_text)
print(f'Report saved to: {report_path}')

if not classification_runs.empty:
    display(classification_runs[['tags.mlflow.runName', 'metrics.accuracy', 'metrics.precision_weighted', 'metrics.recall_weighted', 'metrics.f1_weighted']])
if not regression_runs.empty:
    display(regression_runs[['tags.mlflow.runName', 'metrics.mae', 'metrics.rmse', 'metrics.r2_score']])
